In [ ]:
"""
[황유민] doctor_assignments (의사 배치 이력, 15,000행 · 7열) 정제 노트북.
data/doctor_assignments 를 읽어 clean / quarantine / excluded 로 나눈다. 원본은 읽기만 한다.
선행 노트북: 01_doctors (정제본·격리본을 부모 키로 사용)

정제 로직
1. 모든 컬럼을 문자열로 읽고 직접 형 변환한다.
2. 복구(B15): doctor_id 하이픈 -> 밑줄, valid_from 슬래시 -> 하이픈.
3. 제외: 모든 컬럼이 같은 완전 중복 행은 첫 행만 유지(이 테이블은 0건).
4. 격리(원본 값 보존 + 사유 코드): 필수값 결측(doctor_id), 존재하지 않는 날짜(valid_from 2024-02-30), allocation_pct 음수(명세 1~100),
   valid_from > valid_to, 더미 ID(hospital_department_id = HDP_999999999), PK 중복.
   - allocation_pct 음수(-100)는 절댓값이 명세값(100)과 같아 부호 반전으로 추정되지만, 원천 근거가 없어 복구하지 않고 "복구 후보"로 힌트만 남긴다.
5. FK(B02): hospital_department_id 는 병원별 진료과(읽기 전용)에 존재해야 한다. doctor_id 는 01_doctors 의 정제본 키에 존재해야 하며,
   부모 의사가 격리된 경우는 직접 오염과 구분해 연쇄(FK_PARENT_QUARANTINED)로 기록한다(자기 값은 정상).
6. 정제 후 재검증 + 참고 점검(같은 의사의 배치 기간 겹침, 채용일 이전 배치).
결과: 수신 15,000 = 제외 0 + 격리 1,356(직접 600, 연쇄 756) + 정제본 13,644
"""

import os
import time
from pathlib import Path

import polars as pl

# 경로: 환경변수 CLEAN_DATA / CLEAN_OUT 이 있으면 그것을, 없으면 아래 기본값을 쓴다.
DATA_DIR = Path(os.environ.get("CLEAN_DATA", r"C:\Users\dbals\OneDrive\Desktop\제공 오염 데이터\data"))   # 원본(읽기 전용)
OUT_DIR = Path(os.environ.get("CLEAN_OUT", r"C:\Users\dbals\OneDrive\Desktop\제공 오염 데이터\정제결과"))      # 정제 결과 출력
for sub in ("clean", "quarantine", "excluded", "flags"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

TABLE = "doctor_assignments"
SENTINEL_SUFFIX = "_999999999"  # 오염 데이터의 더미 ID 패턴 (예: APT_999999999)
print("polars", pl.__version__, "| 입력:", DATA_DIR / TABLE, "| 출력:", OUT_DIR)

In [ ]:
# ---- 입출력 도우미 ----
def list_files(table):
    # 엑셀 잠금 파일(~$...)은 데이터가 아니므로 제외
    return sorted(p for p in (DATA_DIR / table).glob("*.csv") if not p.name.startswith("~$"))


def read_table(table, usecols=None):
    """CSV 전체를 문자열로 읽고 원본 파일명/줄 번호(헤더=1)를 붙인다. 빈 값은 null."""
    frames = []
    for f in list_files(table):
        d = pl.read_csv(f, infer_schema=False, columns=usecols)
        d = d.with_columns(pl.lit(f.name).alias("_src_file"),
                           (pl.int_range(pl.len(), dtype=pl.Int64) + 2).alias("_src_line"))
        frames.append(d)
    return pl.concat(frames, how="vertical")


def write_csv_bom(df, path):
    """원본과 같은 형식: UTF-8 BOM, 쉼표 CSV, LF. 파일이 잠겨 있으면 기다렸다가 재시도."""
    for attempt in range(1, 25):
        try:
            with open(path, "wb") as fh:
                fh.write(b"\xef\xbb\xbf")
                df.write_csv(fh, date_format="%Y-%m-%d", datetime_format="%Y-%m-%d %H:%M:%S")
            return
        except PermissionError:
            print(f"  (파일이 다른 프로그램에서 사용 중: {Path(path).name}, {attempt}/24 재시도, 10초 대기)", flush=True)
            time.sleep(10)
    raise PermissionError(f"{path} 를 4분 넘게 쓸 수 없습니다. 이 파일을 연 프로그램을 닫고 다시 실행하세요.")


def norm_key(col):
    """ID 정규화: 앞뒤 공백 제거 + 하이픈을 밑줄로 (B15). 빈 값은 null."""
    v = pl.col(col).str.strip_chars().str.replace_all("-", "_", literal=True)
    return pl.when(v == "").then(None).otherwise(v)


def ext_keys(table, col):
    """FK 검증용 외부 부모 테이블 키(읽기 전용). 정규화 후 존재 여부만 본다."""
    d = read_table(table, usecols=[col]).select(norm_key(col).alias("key")).drop_nulls().unique()
    return d.with_columns(pl.lit("ok").alias("st"))


def upstream_parents(table, key_col):
    """앞 노트북이 만든 부모 정제본/격리본에서 부모 키를 읽는다. st='ok'(정상 부모) / 'q'(격리된 부모)."""
    ok = (pl.read_csv(OUT_DIR / "clean" / f"{table}_clean.csv", infer_schema=False, columns=[key_col])
          .select(pl.col(key_col).alias("key")).with_columns(pl.lit("ok").alias("st")))
    q = (pl.read_csv(OUT_DIR / "quarantine" / f"{table}_quarantine.csv", infer_schema=False, columns=[key_col])
         .select(norm_key(key_col).alias("key")).drop_nulls().unique().with_columns(pl.lit("q").alias("st")))
    return pl.concat([ok, q])

In [ ]:
# ---- 표준 코드 사전 / 원천 코드 매핑 (명세 code_dictionary, legacy_code_mappings) ----
_cd = read_table("code_dictionary")
_lm = read_table("legacy_code_mappings")
STD = {}
for dom, code in zip(_cd["domain"], _cd["standard_code"]):
    STD.setdefault(dom, []).append(code)
MAPS = _lm.select("domain", "source_system", pl.col("raw_code"), pl.col("standard_code"))


# ---- 컬럼 설정 (명세서 기준) ----
def C(name, kind, spec, nn=True, prefix=None, domain=None, maxlen=None, values=None):
    return dict(name=name, kind=kind, spec=spec, nn=nn, prefix=prefix, domain=domain, maxlen=maxlen, values=values)


SRC = C("source_system", "enum", "VARCHAR(8) NOT NULL, LEGACY_A/LEGACY_B", values=["LEGACY_A", "LEGACY_B"])
BAD_REASON = {"id": "ID_FORMAT", "str": "LENGTH_EXCEEDED", "date": "DATE_INVALID", "datetime": "DATE_INVALID",
              "code": "CODE_UNMAPPED", "int": "NOT_NUMERIC", "enum": "ENUM_INVALID"}

COLS = [
    C("assignment_id", "id", "VARCHAR(32) PK NOT NULL (ASN_9자리)", prefix="ASN"),
    C("doctor_id", "id", "VARCHAR(32) NOT NULL, FK->doctors", prefix="DOC"),
    C("hospital_department_id", "id", "VARCHAR(32) NOT NULL, FK->hospital_departments", prefix="HDP"),
    C("valid_from", "date", "DATE NOT NULL, 유효 시작일(포함)"),
    C("valid_to", "date", "DATE NOT NULL, 유효 종료일(포함)"),
    C("allocation_pct", "int", "INT NOT NULL, 1~100 (본 버전은 100)"),
    SRC,
]
PK = "assignment_id"

In [ ]:
# ---- 정규화(복구) : 원본 컬럼은 그대로 두고 __n(정규화값) / __bad(복구 불가 표시) / __fix_*(복구 내역) 컬럼을 추가 ----
def normalize(df, cols):
    df = df.with_columns(pl.int_range(pl.len(), dtype=pl.Int64).alias("_idx"))
    for c in cols:
        n, k = c["name"], c["kind"]
        raw = pl.col(n)
        s = raw.str.strip_chars()
        trimfix = (raw != s).fill_null(False)
        if k == "id":
            v = s.str.replace_all("-", "_", literal=True)
            v = pl.when(v == "").then(None).otherwise(v)
            ok = v.str.contains(rf"^{c['prefix']}_\d{{9}}$")
            df = df.with_columns(v.alias(f"{n}__n"),
                                 (v.is_not_null() & ~ok).fill_null(False).alias(f"{n}__bad"),
                                 raw.str.contains("-", literal=True).fill_null(False).alias(f"{n}__fix_hyphen"),
                                 trimfix.alias(f"{n}__fix_trim"))
        elif k == "str":
            v = pl.when(s == "").then(None).otherwise(s)
            df = df.with_columns(v.alias(f"{n}__n"),
                                 (v.str.len_chars() > c["maxlen"]).fill_null(False).alias(f"{n}__bad"),
                                 trimfix.alias(f"{n}__fix_trim"))
        elif k in ("date", "datetime"):
            v = s.str.replace_all("/", "-", literal=True)
            if k == "date":
                gate, fmt = r"^\d{4}-\d{2}-\d{2}$", "%Y-%m-%d"
                parsed = pl.when(v.str.contains(gate)).then(v.str.to_date(fmt, strict=False)).otherwise(None)
            else:
                gate, fmt = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$", "%Y-%m-%d %H:%M:%S"
                parsed = pl.when(v.str.contains(gate)).then(v.str.to_datetime(fmt, strict=False)).otherwise(None)
            df = df.with_columns(parsed.alias(f"{n}__n"),
                                 raw.str.contains("/", literal=True).fill_null(False).alias(f"{n}__fix_slash"))
            df = df.with_columns(((pl.col(f"{n}__n").is_null()) & s.is_not_null() & (s != "")).alias(f"{n}__bad"))
        elif k == "int":
            parsed = s.cast(pl.Int64, strict=False)
            df = df.with_columns(parsed.alias(f"{n}__n"),
                                 (s.is_not_null() & (s != "") & parsed.is_null()).alias(f"{n}__bad"))
        elif k == "enum":
            v = pl.when(s.is_in(c["values"])).then(s).otherwise(None)
            df = df.with_columns(v.alias(f"{n}__n"),
                                 (s.is_not_null() & (s != "") & v.is_null()).alias(f"{n}__bad"))
        elif k == "code":
            dom = c["domain"]
            m = (MAPS.filter(pl.col("domain") == dom)
                 .select("source_system", pl.col("raw_code").alias(f"{n}__s"), pl.col("standard_code").alias(f"{n}__alias")))
            df = df.with_columns(s.alias(f"{n}__s")).join(m, on=["source_system", f"{n}__s"], how="left")
            ss = pl.col(f"{n}__s")
            is_std = ss.is_in(STD[dom]).fill_null(False)
            v = (pl.when(is_std).then(ss).when(pl.col(f"{n}__alias").is_not_null()).then(pl.col(f"{n}__alias"))
                 .otherwise(None))
            df = df.with_columns(v.alias(f"{n}__n"),
                                 (~is_std & pl.col(f"{n}__alias").is_not_null()).alias(f"{n}__fix_alias"))
            df = df.with_columns((ss.is_not_null() & (ss != "") & pl.col(f"{n}__n").is_null()).alias(f"{n}__bad"))
    return df.sort("_idx")


# ---- 격리 사유 부여 도구 ----
def reason(cond, code):
    return pl.when(cond.fill_null(False)).then(pl.lit(code)).otherwise(pl.lit(None, dtype=pl.String))


def generic_reasons(cols):
    """모든 테이블 공통: 필수값 결측 / 형식·길이·날짜·코드·숫자 오류."""
    out = []
    for c in cols:
        n = c["name"]
        if c["nn"]:
            out.append(reason(pl.col(f"{n}__n").is_null() & ~pl.col(f"{n}__bad"), f"NULL_REQUIRED:{n}"))
        out.append(reason(pl.col(f"{n}__bad"), f"{BAD_REASON[c['kind']]}:{n}"))
    return out


def add_fk(df, col, parents):
    """parents: DataFrame(key, st) st='ok'(정상 부모) / 'q'(격리된 부모). 결과 컬럼 {col}__fk."""
    p = parents.select(pl.col("key").alias(f"{col}__n"), pl.col("st").alias(f"{col}__fk")).unique(f"{col}__n")
    return df.join(p, on=f"{col}__n", how="left").sort("_idx")


def fk_direct(col):
    """부모에 키가 없는 경우: 더미 ID 는 FK_SENTINEL, 그 밖은 FK_NOT_FOUND (직접 오염)."""
    n = pl.col(f"{col}__n")
    return pl.when(n.is_not_null() & pl.col(f"{col}__fk").is_null() & n.str.ends_with(SENTINEL_SUFFIX)) \
        .then(pl.lit(f"FK_SENTINEL:{col}")) \
        .when(n.is_not_null() & pl.col(f"{col}__fk").is_null()).then(pl.lit(f"FK_NOT_FOUND:{col}")) \
        .otherwise(pl.lit(None, dtype=pl.String))


def fk_cascade(col):
    """부모가 격리되어 생긴 FK 위반 (연쇄): 자기 값은 정상."""
    return reason(pl.col(f"{col}__fk") == "q", f"FK_PARENT_QUARANTINED:{col}")


def finalize_reasons(df, direct, cascade):
    lst = pl.concat_list(direct + cascade).list.drop_nulls()
    return df.with_columns(lst.alias("reasons")).with_columns(
        pl.col("reasons").list.len().alias("n_reasons"),
        pl.col("reasons").list.first().alias("reason_primary"),
        pl.col("reasons").list.eval(~pl.element().str.starts_with("FK_PARENT_QUARANTINED")).list.any().alias("has_direct"))


# ---- 복구 힌트 (격리 파일의 repair_hint 컬럼) ----
HINT = {
    "NULL_REQUIRED": "원천 값이 비어 있어 복구 불가",
    "ID_FORMAT": "ID 형식이 명세와 달라 복구 불가",
    "DATE_INVALID": "존재하지 않는 날짜라 복구 불가",
    "CODE_UNMAPPED": "매핑표에 없는 값이라 추측 금지(B14)",
    "NOT_NUMERIC": "숫자가 아니어서 복구 불가",
    "LENGTH_EXCEEDED": "길이 초과로 복구 불가",
    "ENUM_INVALID": "허용값이 아니라 복구 불가",
    "PK_DUP": "PK 중복: 어느 행이 정본인지 근거 없음",
    "NEG_VALUE": "부호 반전 추정(절댓값이 명세 범위와 일치). 팀 합의 시 복구 후보",
    "RANGE": "허용 범위 밖 값",
    "ORDER": "순서 역전: 어느 쪽이 틀렸는지 근거 없음",
    "FK_SENTINEL": "더미 ID라 복구 불가",
    "FK_NOT_FOUND": "부모 키 없음(부모 PK 결측 가능). 부모 정제 후 재검사",
    "FK_PARENT_QUARANTINED": "부모 격리로 인한 연쇄. 부모 복구 시 재검사",
    "B04_OUT_OF_SESSION": "슬롯이 운영 구간 시간 밖",
    "B03_OUT_OF_RANGE": "배치 유효기간과 운영일 불일치",
    "B03_DEPT_MISMATCH": "세션과 배치의 진료과 불일치",
    "B07_RESERVED_NO_APPT": "예약접수인데 예약 ID 없음",
    "B07_WALKIN_HAS_APPT": "현장접수인데 예약 ID 있음",
}
HINT_FULL = {
    "CODE_UNMAPPED:visit_type": "appointment_id 유무로 유추 가능하나 추측 금지(B07 참고)",
    "CODE_UNMAPPED:outcome": "진료(encounter) 연결 유무로 유추 가능하나 추측 금지(B08 참고)",
}


# ---- 한 테이블 처리: 읽기 -> 정규화(복구) -> 중복 제외 -> 규칙(격리 사유) -> clean / quarantine 저장 ----
def run_table(t, cols, pk, rules):
    t0 = time.time()
    names = [c["name"] for c in cols]
    raw = read_table(t)
    df = normalize(raw, cols)

    # 완전 중복 제외 (원본 값 전체가 같은 행의 재전송, 첫 행만 유지)
    df = df.with_columns(pl.struct(names).is_first_distinct().alias("_first"))
    dup = df.filter(~pl.col("_first"))
    df = df.filter(pl.col("_first"))
    write_csv_bom(dup.select(names + ["_src_file", "_src_line"]), OUT_DIR / "excluded" / f"{t}_duplicates_excluded.csv")

    # 격리 사유: 공통 규칙 + PK 중복 + 테이블별 규칙
    direct = generic_reasons(cols)
    direct.append(reason(pl.col(f"{pk}__n").is_not_null() & pl.col(f"{pk}__n").is_duplicated(), f"PK_DUP:{pk}"))
    df, extra_direct, cascade = rules(df)
    direct += extra_direct
    df = finalize_reasons(df, direct, cascade)

    # 분리: 사유 없음 -> clean / 있음 -> quarantine (원본 값 그대로 + 사유 + 복구 힌트)
    q = df.filter(pl.col("n_reasons") > 0)
    c = df.filter(pl.col("n_reasons") == 0)
    clean = c.select([pl.col(f"{n}__n").alias(n) for n in names])
    primary_key_part = pl.col("reason_primary").str.split(":").list.first()
    quar = q.select(
        names + ["_src_file", "_src_line",
                 pl.col("reasons").list.join("|").alias("reason_codes"),
                 pl.col("reason_primary"),
                 pl.when(pl.col("has_direct")).then(pl.lit("직접")).otherwise(pl.lit("연쇄")).alias("quarantine_type"),
                 pl.coalesce(pl.col("reason_primary").replace_strict(HINT_FULL, default=None, return_dtype=pl.String),
                             primary_key_part.replace_strict(HINT, default="", return_dtype=pl.String)).alias("repair_hint")])
    write_csv_bom(clean, OUT_DIR / "clean" / f"{t}_clean.csv")
    write_csv_bom(quar, OUT_DIR / "quarantine" / f"{t}_quarantine.csv")
    quar_keys = q.select(pl.col(f"{pk}__n").alias("key")).drop_nulls().unique().with_columns(pl.lit("q").alias("st"))

    # 요약
    n_direct = int(q.filter(pl.col("has_direct")).height)
    n_casc = int(q.filter(~pl.col("has_direct")).height)
    print(f"[{t}] 수신 {raw.height:,} = 제외(중복) {dup.height:,} + 격리(직접) {n_direct:,} + 격리(연쇄) {n_casc:,} + 정제본 {c.height:,}"
          f"  | 합계 일치: {raw.height == dup.height + n_direct + n_casc + c.height}  ({time.time() - t0:.1f}s)")
    print("\n격리 사유별 행 수 (한 행에 사유가 여러 개일 수 있음):")
    print(q.select(pl.col("reasons").explode().alias("사유")).group_by("사유").len().sort("len", descending=True))
    print("\n복구(보정) 내역: 중복 제외 후 행 기준")
    for col in df.columns:
        if "__fix_" in col:
            base, typ = col.split("__fix_")
            tot = int(df.get_column(col).sum())
            if tot:
                print(f"  {base:28s} {typ:8s} {tot:>8,}건 (정제본에 반영 {int(c.get_column(col).sum()):,})")
    return clean, quar_keys


# ---- 정제 후 재검증: 정제본에 같은 규칙을 다시 적용해 위반 건수를 센다 ----
def validate_common(clean, cols, pk):
    out = [(f"PK 결측 ({pk})", int(clean[pk].null_count())),
           (f"PK 중복 ({pk})", int(clean[pk].is_duplicated().sum())),
           ("완전 중복 행", int(clean.is_duplicated().sum()))]
    for c in cols:
        n = c["name"]
        if c["nn"] and n != pk:
            out.append((f"NOT NULL 위반 ({n})", int(clean[n].null_count())))
        if c["kind"] == "id":
            out.append((f"ID 형식 위반 ({n})", chk(clean, pl.col(n).is_not_null() & ~pl.col(n).str.contains(rf"^{c['prefix']}_\d{{9}}$"))))
        if c["kind"] == "code":
            out.append((f"표준 코드 아님 ({n})", chk(clean, pl.col(n).is_not_null() & ~pl.col(n).is_in(STD[c["domain"]]))))
        if c["kind"] == "enum":
            out.append((f"허용값 아님 ({n})", chk(clean, ~pl.col(n).is_in(c["values"]))))
    return out


def chk(clean, expr):
    return int(clean.select(expr.fill_null(False).sum()).item())


def fkcheck(clean, col, keys):
    s = clean.get_column(col).drop_nulls()
    return (f"FK 위반 ({col})", int((~s.is_in(keys.implode())).sum()))


def show_validation(out):
    for name, n in out:
        print(f"  {name:40s} {n:>8,}")
    print("정제본 위반 합계:", sum(n for _, n in out), "건")

In [ ]:
# ---- 이 테이블 전용 규칙 ----
HDP_KEYS = ext_keys("hospital_departments", "hospital_department_id")   # 부모: 병원별 진료과(읽기 전용)
DOCTOR_KEYS = upstream_parents("doctors", "doctor_id")                    # 부모: 01_doctors 결과(정상 'ok' / 격리 'q')


def rules(df):
    """doctor_assignments 전용 규칙: 기간 순서, allocation_pct 범위, FK(의사·병원 진료과)."""
    df = add_fk(df, "doctor_id", DOCTOR_KEYS)
    df = add_fk(df, "hospital_department_id", HDP_KEYS)
    direct = [reason(pl.col("valid_from__n") > pl.col("valid_to__n"), "ORDER:valid_from>valid_to"),
              reason(pl.col("allocation_pct__n") < 0, "NEG_VALUE:allocation_pct"),
              reason((pl.col("allocation_pct__n") == 0) | (pl.col("allocation_pct__n") > 100), "RANGE:allocation_pct"),
              fk_direct("doctor_id"), fk_direct("hospital_department_id")]
    return df, direct, [fk_cascade("doctor_id")]

In [ ]:
# ---- 실행: 정제본 / 격리본 / 중복 제외본을 파일로 저장 ----
clean, quar_keys = run_table(TABLE, COLS, PK, rules)
clean.head(5)

In [ ]:
# ---- 정제 후 재검증 (위반 건수는 모두 0이어야 한다) ----
DOCTOR_CLEAN_IDS = DOCTOR_KEYS.filter(pl.col("st") == "ok")["key"]
out = validate_common(clean, COLS, PK)
out.append(fkcheck(clean, "doctor_id", DOCTOR_CLEAN_IDS))
out.append(fkcheck(clean, "hospital_department_id", HDP_KEYS["key"]))
out.append(("valid_from > valid_to", chk(clean, pl.col("valid_from") > pl.col("valid_to"))))
out.append(("allocation_pct 1~100 밖", chk(clean, (pl.col("allocation_pct") < 1) | (pl.col("allocation_pct") > 100))))
show_validation(out)

In [ ]:
# 참고 점검(격리 아님): 같은 의사의 배치 기간 겹침 / 배치 시작일이 채용일보다 이른 건
a = clean.sort(["doctor_id", "valid_from"]).with_columns(pl.col("valid_to").shift(1).over("doctor_id").alias("prev_to"))
overlap = int((a["valid_from"] <= a["prev_to"]).fill_null(False).sum())
hire = (pl.read_csv(OUT_DIR / "clean" / "doctors_clean.csv", infer_schema=False, columns=["doctor_id", "hire_date"])
        .with_columns(pl.col("hire_date").str.to_date("%Y-%m-%d")))
before_hire = int(a.join(hire, on="doctor_id", how="left").select((pl.col("valid_from") < pl.col("hire_date")).fill_null(False).sum()).item())
print(f"정제본 {a.height:,}건 중 배치 기간 겹침 {overlap:,}건, 채용일 이전 배치 {before_hire:,}건")